In [34]:
import ssl
import certifi
import urllib.request

ssl_context = ssl.create_default_context(cafile=certifi.where())
opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ssl_context))
urllib.request.install_opener(opener)

import torch
import torch.nn as nn
import torchvision.models as models

In [35]:
torch.hub.set_dir('/tmp/torch_hub')

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(512, 2)

In [36]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [37]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [38]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [39]:
dataset = ImageFolder('/Users/aymanaghel/Desktop/Ivy CV/photos', transform=transform)
print(dataset.classes)
print(len(dataset))

['Ivy photo', 'Not Ivy']
1452


In [40]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(train_size, test_size)

1161 291


In [41]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [42]:
for epoch in range(8):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.4133
Epoch 2, Loss: 0.1371
Epoch 3, Loss: 0.0588
Epoch 4, Loss: 0.0348
Epoch 5, Loss: 0.0262
Epoch 6, Loss: 0.0175
Epoch 7, Loss: 0.0207
Epoch 8, Loss: 0.0138


In [43]:
torch.save(model.state_dict(), 'ivy_classifier.pth')

In [44]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 98.63%


In [45]:
from PIL import Image

test_img = Image.open("your_image.jpg")
test_tensor = transform(test_img).unsqueeze(0)

model.eval()
with torch.no_grad():
    output = model(test_tensor)
    _, predicted = torch.max(output, 1)

print(dataset.classes[predicted.item()])

FileNotFoundError: [Errno 2] No such file or directory: 'your_image.jpg'